# Story 1: Submit a Long-running Function and Come Back Later
**Who:** A scientist with a Python function which runs a simulation for hours, for example a molecular dynamics run
at a given temperature.

**What they want:** Run the function for a few temperatures on the cluster, log out, and pick up the results the next
morning, directly as Python objects.

**Why plain SLURM is painful:** The function has to be wrapped in a Python script which reads its inputs from files,
the script in a batch script, the batch script is submitted with `sbatch`, the job IDs are noted down and checked with
`squeue`, and finally the output files are parsed back into Python. A single function call turns into three files and
a manual bookkeeping exercise.

**How executorlib helps:** The `SlurmClusterExecutor` submits every function call as its own SLURM job and stores
inputs and outputs in a cache directory on the shared file system. Because the state lives on disk rather than in the
Python process, you can disconnect, check the status from any Python process, reconnect later and cancel jobs you no
longer need.

This notebook runs on the **login node** of a SLURM cluster - no allocation is required.

## Cluster-specific settings
**Adapt the following cell to your cluster.** It collects everything which depends on the cluster rather than on your science. The values below are the ones used to test this notebook on a single-node SLURM test cluster. The submission template is a [Jinja2](https://jinja.palletsprojects.com) template which executorlib renders for every function call: add the lines your cluster requires, for example `#SBATCH --partition=<name>`, `#SBATCH --account=<project>` or `module load ...` commands before `{{command}}`. The [executorlib-cluster-setup](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#setup-with-claude-code) skill for Claude Code can analyse your cluster and generate a tested template.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster.
submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env=L
#SBATCH --ntasks={{cores}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency_list %}
#SBATCH --dependency=afterok:{{ dependency_list | join(',') }}
{%- endif %}

{{command}}
"""

## Step 1: Write the function
The function is plain Python. It does not know anything about SLURM.

In [ ]:
import time


def simulate(temperature, n_steps=500):
    # Stand-in for an hours-long simulation: the run time grows with the number of steps.
    time.sleep(n_steps / 100)
    return {"temperature": temperature, "energy": -3.5 + 0.001 * temperature}

## Step 2: Submit and disconnect
Every call of `submit()` serializes the function and its arguments to the `cache_directory`, renders the submission
template and calls `sbatch`. The `resource_dict` defined for the executor applies to all calls - here the maximum run
time `run_time_max` in seconds, which the template converts to `#SBATCH --time`.

Instead of a `with` statement, which waits for all jobs to finish, the executor is shut down with
`shutdown(wait=False, cancel_futures=False)`. This returns immediately and leaves the jobs running in SLURM.

In [ ]:
from executorlib import SlurmClusterExecutor

cache_directory = "./story1_cache"
temperature_lst = [300, 600, 900]

exe = SlurmClusterExecutor(
    cache_directory=cache_directory,
    resource_dict={"submission_template": submission_template, "run_time_max": 600},
)
future_lst = [exe.submit(simulate, temperature) for temperature in temperature_lst]
exe.shutdown(wait=False, cancel_futures=False)

The future objects of this Python process are now detached. They are marked as cancelled, because this process no
longer follows the jobs - the jobs themselves keep running. At this point you could close the notebook and log out.

In [ ]:
[future.cancelled() for future in future_lst]

## Step 3: Check the status from any Python process
`get_cache_data_queue()` combines the content of the cache directory with the output of `squeue`. It only needs the
`cache_directory`, so it works from a new Python process as well. The `status` is `pending` or `running` while the job
is in the queue, `finished` once the result was written and `aborted` when the job left the queue without a result -
see [Story 5](story-5-recover-failed-jobs.ipynb) for how to handle that case.

In [ ]:
import pandas
from executorlib import get_cache_data_queue

df = pandas.DataFrame(get_cache_data_queue(cache_directory=cache_directory, queue_type="slurm"))
df[["input_args", "queue_id", "status"]]

## Step 4: Reconnect and collect the results
Later - after lunch, the next morning, or from a fresh Python process - submit the same function calls with the same
`cache_directory`. executorlib identifies each call by a hash of the function and its arguments:

* calls which already finished are loaded from the cache,
* calls whose job is still in the queue are re-attached to that job instead of being submitted a second time,
* only calls which are unknown to the cache are submitted as new jobs.

In a new Python process define the function with exactly the same code, otherwise the hash changes and new jobs are
submitted.

In [ ]:
with SlurmClusterExecutor(
    cache_directory=cache_directory,
    resource_dict={"submission_template": submission_template, "run_time_max": 600},
) as exe:
    future_lst = [exe.submit(simulate, temperature) for temperature in temperature_lst]
    result_lst = [future.result() for future in future_lst]

result_lst

In [ ]:
df = pandas.DataFrame(get_cache_data_queue(cache_directory=cache_directory, queue_type="slurm"))
df[["input_args", "queue_id", "status", "runtime"]]

## Step 5: Cancel jobs you no longer need
Mistakes happen: here a run with one hundred times too many steps is submitted and the session is disconnected again.

In [ ]:
exe = SlurmClusterExecutor(
    cache_directory=cache_directory,
    resource_dict={"submission_template": submission_template, "run_time_max": 600},
)
future_wrong = exe.submit(simulate, 1200, n_steps=50000)  # typo: should have been n_steps=500
exe.shutdown(wait=False, cancel_futures=False)

df = pandas.DataFrame(get_cache_data_queue(cache_directory=cache_directory, queue_type="slurm"))
df[["input_args", "input_kwargs", "queue_id", "status"]]

`terminate_tasks_in_cache()` calls `scancel` for every unfinished job recorded in the cache directory and removes its
input file, so a later resubmission starts from scratch. Finished results remain in the cache. To cancel a single
call use `terminate_task_in_cache()` with the `cache_key` of the call, which is the file name of its cache file without
the `_i.h5` suffix.

In [ ]:
from executorlib import terminate_tasks_in_cache

terminate_tasks_in_cache(cache_directory=cache_directory, backend="slurm")

df = pandas.DataFrame(get_cache_data_queue(cache_directory=cache_directory, queue_type="slurm"))
df[["input_args", "input_kwargs", "queue_id", "status"]]

## Clean up
executorlib never deletes the cache directory, since it might contain results which took days to compute. Remove it
once the results are stored elsewhere - the cache format is designed for caching, not for long-term storage.

In [ ]:
import shutil

shutil.rmtree(cache_directory, ignore_errors=True)

## What you learned
* The `SlurmClusterExecutor` submits each function call as a SLURM job, configured by a `submission_template` and the
  `resource_dict`.
* `shutdown(wait=False, cancel_futures=False)` disconnects the Python process while the jobs keep running.
* Resubmitting the same calls with the same `cache_directory` reconnects to running jobs and reloads finished results.
* `get_cache_data_queue()` reports the status of all calls, `terminate_tasks_in_cache()` cancels unfinished ones.

**Reference:** [HPC Cluster Executor](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html) (submission templates, pysqa configuration directory,
[disconnecting and reconnecting](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#disconnecting-and-reconnecting),
[cleaning the cache](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#cleaning-cache)) and the
[resource dictionary](https://executorlib.readthedocs.io/en/latest/trouble_shooting.html#resource-dictionary).